In [2]:
import requests
import pandas as pd
import unicodedata

# 1. Os 10 estados com mais municípios disponíveis
top_10_estados = ['sp', 'ba', 'rj', 'se', 'ma', 'pr', 'to', 'rs', 'mg', 'ce']
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

print("1. Coletando raspadores ativos no GitHub...")
dados_github = []

for estado in top_10_estados:
    url = f"https://api.github.com/repos/okfn-brasil/querido-diario/contents/data_collection/gazette/spiders/{estado}"
    resp = requests.get(url, headers=headers)

    if resp.status_code == 200:
        for item in resp.json():
            nome_arquivo = item.get('name', '')
            if nome_arquivo.endswith('.py') and nome_arquivo != '__init__.py':
                nome_sem_extensao = nome_arquivo.replace('.py', '')
                uf = nome_sem_extensao[:2].upper()
                cidade_clean = nome_sem_extensao[3:]
                dados_github.append({'estado': uf, 'cidade_github': cidade_clean})

df_github = pd.DataFrame(dados_github)
print(f"-> Encontrados {len(df_github)} raspadores no Top 10 estados.\n")

# 2. Coletando a base oficial do IBGE
print("2. Coletando códigos oficiais na API do IBGE...")
url_ibge = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
resp_ibge = requests.get(url_ibge)

dados_ibge = []
for m in resp_ibge.json():
    estado_sigla = None

    # Navegação segura: Tenta a hierarquia antiga primeiro
    if isinstance(m.get('microrregiao'), dict):
        estado_sigla = m['microrregiao']['mesorregiao']['UF']['sigla']
    # Se falhar, tenta a nova hierarquia (Regiões Imediatas)
    elif isinstance(m.get('regiao-imediata'), dict):
        estado_sigla = m['regiao-imediata']['regiao-intermediaria']['UF']['sigla']

    # Só adiciona se conseguiu achar o Estado
    if estado_sigla:
        dados_ibge.append({
            'ibge_id': str(m['id']),
            'nome_oficial': m['nome'],
            'estado': estado_sigla
        })

df_ibge = pd.DataFrame(dados_ibge)

# 3. Função para normalizar o texto do IBGE
def padronizar_nome(nome):
    nome_sem_acento = ''.join(c for c in unicodedata.normalize('NFD', nome) if unicodedata.category(c) != 'Mn')
    return nome_sem_acento.lower().replace(' ', '_').replace('-', '_').replace("'", "")

df_ibge['cidade_clean'] = df_ibge['nome_oficial'].apply(padronizar_nome)

# 4. Cruzando as duas bases
print("3. Cruzando as bases de dados...")
df_final = pd.merge(df_github, df_ibge,
                    left_on=['estado', 'cidade_github'],
                    right_on=['estado', 'cidade_clean'],
                    how='inner')

cidades_perdidas = len(df_github) - len(df_final)
if cidades_perdidas > 0:
    print(f"\n[Aviso] {cidades_perdidas} municípios não cruzaram perfeitamente devido a diferenças de grafia complexas.")

print(f"\nSucesso! Códigos IBGE de {len(df_final)} municípios extraídos.")

# 5. Lista pronta para o loop de extração
lista_codigos_ibge = df_final['ibge_id'].tolist()

print("\nAmostra dos dados consolidados:")
print(df_final[['estado', 'nome_oficial', 'ibge_id']].head(5))

1. Coletando raspadores ativos no GitHub...
-> Encontrados 387 raspadores no Top 10 estados.

2. Coletando códigos oficiais na API do IBGE...
3. Cruzando as bases de dados...

[Aviso] 16 municípios não cruzaram perfeitamente devido a diferenças de grafia complexas.

Sucesso! Códigos IBGE de 371 municípios extraídos.

Amostra dos dados consolidados:
  estado        nome_oficial  ibge_id
0     SP              Adolfo  3500204
1     SP               Aguaí  3500303
2     SP      Águas da Prata  3500402
3     SP  Águas de São Pedro  3500600
4     SP         Alto Alegre  3501103
